# Example 4: Loading a Subset of Transducers from DataRES

This notebook shows how to load a physical room from the **DataRES** dataset while selecting only a subset of the available transducers.

In many cases you may not need all the measured positions – for instance, you might want to test your RES with fewer microphones or loudspeakers. The `PhRoom_dataset` class allows you to specify which indices to load for the microphone and loudspeaker arrays.

Additionally, PyRES provides a **dataset API** (`dataset_api.py`) to inspect the dataset contents before loading: you can query the number of transducers, their labels, and other low‑level information.

## 1. Imports and Path Setup

In [ ]:
import sys
import os
# Add parent directory to path (so that PyRES can be imported)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import matplotlib.pyplot as plt
from PyRES.physical_room import PhRoom_dataset
from PyRES.dataset_api import get_hl_info, get_ll_info, get_transducer_number

## 2. Time–Frequency Parameters

These are the usual parameters for FFT processing.

In [ ]:
samplerate = 48000          # Hz
nfft = samplerate * 3       # FFT size (3 seconds)
alias_decay_db = 0          # No extra anti‑aliasing decay

## 3. Inspect the Dataset with the API

Before loading the room, we can use the dataset API to explore what is available. The function `get_hl_info()` returns high‑level information about the room, including the directory where its data is stored. Then `get_ll_info()` reads the low‑level JSON file containing transducer numbers, positions, and other metadata. Finally, `get_transducer_number()` extracts the counts for each transducer group.

In [ ]:
dataset_directory = './dataRES'   # Change this to your actual DataRES folder
room_name = 'ImmersiveLab'

print(f"\nInspecting room '{room_name}' in dataset '{dataset_directory}'...")

high_level_info = get_hl_info(ds_dir=dataset_directory, room=room_name)
room_directory = high_level_info['RoomDirectory']
print(f"Room directory: {room_directory}")

low_level_info = get_ll_info(ds_dir=dataset_directory, room_dir=room_directory)
transducer_number, _ = get_transducer_number(ll_info=low_level_info)

print(f"\nNumber of transducers available in '{room_name}':")
print(f"  Stage emitters: {transducer_number['stg']}")
print(f"  System microphones: {transducer_number['mcs']}")
print(f"  System loudspeakers: {transducer_number['lds']}")
print(f"  Audience receivers: {transducer_number['aud']}")

## 4. Select a Subset of Transducers

Now we decide which specific microphones and loudspeakers we want to load. Indices are zero‑based and refer to the order in the dataset. For example, if there are 12 microphones available, we might load only indices 0, 1, and 3.

In [ ]:
mcs_indices = [0, 1, 3]     # Load microphones with indices 0, 1, 3
lds_indices = [0, 2, 8, 11] # Load loudspeakers with indices 0, 2, 8, 11

print(f"Selected microphone indices: {mcs_indices}")
print(f"Selected loudspeaker indices: {lds_indices}")

## 5. Load the Room with the Subset

Pass the index lists to the `PhRoom_dataset` constructor via the `mcs_idx` and `lds_idx` arguments. The other transducer groups (stage emitters and audience receivers) are always loaded completely.

In [ ]:
physical_room = PhRoom_dataset(
    fs=samplerate,
    nfft=nfft,
    alias_decay_db=alias_decay_db,
    dataset_directory=dataset_directory,
    room_name=room_name,
    mcs_idx=mcs_indices,
    lds_idx=lds_indices
)

print(f"\nNumber of transducers actually loaded:")
print(f"  Stage emitters: {physical_room.transducer_number['stg']}")
print(f"  System microphones: {physical_room.transducer_number['mcs']}")
print(f"  System loudspeakers: {physical_room.transducer_number['lds']}")
print(f"  Audience receivers: {physical_room.transducer_number['aud']}")

print(f"\nTransducer positions (x, y, z in meters):")
print(f"  Stage emitters: \n{physical_room.transducer_positions['stg']}")
print(f"  System microphones: \n{physical_room.transducer_positions['mcs']}")
print(f"  System loudspeakers: \n{physical_room.transducer_positions['lds']}")
print(f"  Audience receivers: \n{physical_room.transducer_positions['aud']}")

## 6. Visualise the Setup

The `plot_setup()` method shows the positions of all loaded transducers. Notice that only the selected microphones and loudspeakers appear.

In [ ]:
physical_room.plot_setup()
plt.show()

## 7. Plot Coupling and Direct‑to‑Reverberant Ratio

These plots are computed only for the loaded subset of transducers, so they reflect the reduced set.

In [ ]:
physical_room.plot_coupling()
plt.show()

physical_room.plot_DRR()
plt.show()

## 8. Conclusion

You have successfully loaded a measured room from DataRES while selecting only a subset of microphones and loudspeakers. This is useful when you want to experiment with different array configurations without reloading the entire dataset.

The dataset API (`dataset_api.py`) provides a convenient way to explore the contents of DataRES before loading. For further details, refer to the documentation in `PyRES/physical_room.py` and `PyRES/dataset_api.py`.